In [16]:
from rdkit.Chem import rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
import os
from PIL import Image
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
import ipywidgets as widgets
from IPython.display import display, clear_output
from IPython.display import Image as IPyImage, display
from rdkit.Chem.Draw import rdMolDraw2D

import re
from io import BytesIO
import base64
import random

In [ ]:
# Read and parse the file
filename = "result.txt"
data = []

with open(filename, "r") as f:
    count = 0
    for line in f:
        if (count < 100000):
            count += 1
        else:
            break
        split_line = line.split("|")
        number = split_line[-1].strip()
        smiles = split_line[0].strip()
        data.append({"smiles": smiles, "number": number})


df = pd.DataFrame(data)
unique_numbers = sorted(df["number"].unique())

def remove_atom_mapping(smarts):
    import re
    return re.sub(r":\d+", "", smarts)

def remove_spectators(reaction_smiles):
    split = reaction_smiles.split(">>")
    reactants = set(split[0].split("."))
    products = set(split[1].split("."))
    if reactants == products:
        return reaction_smiles
    unique_reactants = reactants - products
    unique_products = products - reactants
    return '.'.join(unique_reactants) + '>>' + '.'.join(unique_products)

#Dropdown widget
number_box = widgets.Text(
    value=unique_numbers[0],
    description="Number:",
    placeholder="Enter number"
)
def draw_smiles_for_number(selected_number):
    subset = df[df["number"] == selected_number]
    rxns = []
    for s in subset["smiles"]:
        try:
            clean_smarts = remove_atom_mapping(s)
            # clean_smarts = remove_spectators(remove_atom_mapping(s))
            rxn = rdChemReactions.ReactionFromSmarts(clean_smarts, useSmiles=True)
            if rxn is not None and rxn.GetNumReactantTemplates() > 0:
                rxns.append(rxn)
            else:
                print(f"Empty or invalid reaction: {s}")
        except Exception as e:
            print(f"Failed to parse: {s} ({e})")
    if rxns:
        images = []
        for rxn in rxns:        
            new_rxn = rdChemReactions.ChemicalReaction()
        
            for i in range(rxn.GetNumReactantTemplates()):
                new_rxn.AddReactantTemplate(
                    Chem.RemoveHs(rxn.GetReactantTemplate(i))
                )
        
            for i in range(rxn.GetNumProductTemplates()):
                new_rxn.AddProductTemplate(
                    Chem.RemoveHs(rxn.GetProductTemplate(i))
                )
            img = Draw.ReactionToImage(new_rxn, subImgSize=(800, 500))
            if img.mode != "RGB":
                img = img.convert("RGB")
            images.append(img)
        if not images:
            print("No images to display.")
            return
        # Use the actual size of the first image
        img_w, img_h = images[0].size
        per_row = 1
        rows = (len(images) + per_row - 1) // per_row
        grid_img = Image.new('RGB', (per_row * img_w, rows * img_h), (255, 255, 255))
        for idx, img in enumerate(images):
            x = (idx % per_row) * img_w
            y = (idx // per_row) * img_h
            grid_img.paste(img, (x, y))
        display(grid_img)
        grid_img.save("temp.png")

    else:
        print("No valid reactions found for this number.")

widgets.interact(draw_smiles_for_number, selected_number=number_box)



interactive(children=(Text(value='1', description='Number:', placeholder='Enter number'), Output()), _dom_clas…

In [ ]:
untemplated = "untemplated.txt"
untemplated_data = []

with open(untemplated, "r") as f:
    for line in f:
        #split_line = line.split(" ")
        split_line = line.split("|")
        smiles = split_line[0].strip()
        untemplated_data.append({"smiles": smiles})


untemplated_df = pd.DataFrame(untemplated_data)

In [ ]:
batch_size = 10

def show_random_batch(b=None):
    clear_output(wait=True)
    # sample 5 random indices without replacement
    if len(untemplated_df) < batch_size:
        sample_indices = list(range(len(untemplated_df)))
    else:
        sample_indices = random.sample(range(len(untemplated_df)), batch_size)
        
    smiles_list = untemplated_df['smiles'].iloc[sample_indices]
    max_len = 100  # adjust as needed
    for idx, s in smiles_list.items():
        s = str(s)
        print(f"{idx}: {s[:max_len]}{'...' if len(s) > max_len else ''}")
    
    rxns = []
    for s in smiles_list:
        try:
            rxn = rdChemReactions.ReactionFromSmarts(remove_atom_mapping(s), useSmiles=True)
            if rxn is not None:
                rxns.append(rxn)
        except Exception as e:
            print(f"Failed to parse: {s} ({e})")
    
    if rxns:
        images = []
        for rxn in rxns:
            img = Draw.ReactionToImage(rxn, subImgSize=(800, 500))
            if img.mode != "RGB":
                img = img.convert("RGB")
            images.append(img)
        if not images:
            print("No images to display.")
            return
        img_w, img_h = images[0].size
        per_row = 1
        rows = (len(images) + per_row - 1) // per_row
        grid_img = Image.new('RGB', (per_row * img_w, rows * img_h), (255, 255, 255))
        for idx, img in enumerate(images):
            x = (idx % per_row) * img_w
            y = (idx // per_row) * img_h
            grid_img.paste(img, (x, y))
            
        display(grid_img)
        display(next_button)
    else:
        print("No valid reactions found to display.")

next_button = widgets.Button(description="Next 5 Random")
next_button.on_click(show_random_batch)


# Show the first random batch and button
show_random_batch()



In [ ]:
test = "templated.txt"

data_test = []

with open(test, "r") as f:
    for line in f:
        split_line = line.split("|")
        number = split_line[-1].strip()
        smiles = split_line[0].strip()
        data_test.append({"smiles": smiles, "number": number})


df_test = pd.DataFrame(data_test)
unique_numbers = sorted(df_test["number"].unique())

In [28]:
# Get unique numbers for dropdown
def remove_spectators(reaction_smiles):
    split = reaction_smiles.split(">>")
    reactants = set(split[0].split("."))
    products = set(split[1].split("."))
    if reactants == products:
        return reaction_smiles
    unique_reactants = reactants - products
    unique_products = products - reactants
    return '.'.join(unique_reactants) + '>>' + '.'.join(unique_products)

# Dropdown widget
number_box = widgets.Text(
    value=unique_numbers[0],
    description="Number:",
    placeholder="Enter number"
)

def draw_smiles_for_number(selected_number, save_path = "images/temp.png"):
    subset = df_test[df_test["number"] == selected_number]
    rxns = []
    for s in subset["smiles"]:
        try:
            clean_smarts = remove_atom_mapping(s)
            rxn = rdChemReactions.ReactionFromSmarts(clean_smarts, useSmiles=True)
            if rxn is not None and rxn.GetNumReactantTemplates() > 0:
                rxns.append(rxn)
            else:
                print(f"Empty or invalid reaction: {clean_smarts}")
        except Exception as e:
            print(f"Failed to parse: {s} ({e})")

    if rxns:
        images = []
        for rxn in rxns:
            img = Draw.ReactionToImage(rxn, subImgSize=(1000, 300))
            if img.mode != "RGB":
                img = img.convert("RGB")
            images.append(img)

        if not images:
            print("No images to display.")
            return

        img_w, img_h = images[0].size
        per_row = 1
        rows = (len(images) + per_row - 1) // per_row
        grid_img = Image.new('RGB', (per_row * img_w, rows * img_h), (255, 255, 255))
        for idx, img in enumerate(images):
            x = (idx % per_row) * img_w
            y = (idx // per_row) * img_h
            grid_img.paste(img, (x, y))

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            grid_img.save(save_path, format="PNG")
            print(f"Saved to: {save_path}")
        else:
            print("Set save_path to write the PNG to disk.")
    else:
        print("No valid reactions found for this number.")

widgets.interact(draw_smiles_for_number, selected_number=number_box)

interactive(children=(Text(value='1', description='Number:', placeholder='Enter number'), Text(value='images/t…

<function __main__.draw_smiles_for_number(selected_number, save_path='images/temp.png')>